<div style="padding: 150px;color:white;margin:10;border-radius:8px;overflow:hidden;background-image: url(https://d3rw207pwvlq3a.cloudfront.net/attachments/000/049/113/original/shutterstock_217626931_%281%29.jpg?1551317344);background-position: 50% 50%"></div>

<p style="padding-bottom: 10px;
          margin-bottom: 10px;
          font-size: 50px;
          font-weight: bold;
          color: black; 
          text-align: left;
          font-family: Poppins"><b>📝 Kaggle - LLM Science Exam 👨‍🔬 </b>
<p style="font-size: 22px; color: gray; text-align: left;">Can we build a model that is up to this Science Exam Challenge?</p>
<hr style="height:2px;border-width:0;color:gray;background-color:gray;box-shadow: 0px 2.5px 5px rgba(0, 0, 0, 0.2);">

<h3 style="border-bottom: 1px solid #ccc;
            padding-bottom: 10px;
            margin-bottom: 10px;
            font-size: 16px;
            font-weight: bold;
            color: black;">Table of Contents</h3>
    
- [Introduction](#intro)<br><br>
- [Exploratory Data Analysis](#eda)<br><br>
- [Modeling](#model)<br><br>
- [Conclusion](#conclusion)<br><br>

<h1 id = 'intro' style="border-bottom: 1px solid #ccc;
                        padding-bottom: 10px;
                        margin-bottom: 10px;
                        font-size: 38px;
                        font-weight: bold;
                        color: black;
                        font-family: Poppins">Introduction</h1>

<p style="font-size: 20px">Welcome to the Kaggle - LLM Science Exam competition! 
<br><br>
In this competition, we are going to develop a model that is able to answer difficult science-based questions <b>written by</b> a Large Language Model. <br><br>
The dataset for this challenge was generated by giving GPT 3.5 snippets of text on a range of scientific topics pulled from wikipedia, and asking it to write a multiple choice question and providing the correct answer. <br><br>
For this competition, we will submit a csv file containing an id number and at least three lables as predictions for questions. <br><br>
The evauation metric is the Mean Average Precision at 3 ($MAP@3$), given by the following equation: <br><br><br></p>
<p style="font-size: 24px">
\[
MAP@3 = \frac{1}{U} \sum_{u=1}^{U} \sum_{k=1}^{min(n,3)} P(k) \times rel(k)
\]
<br><br><br></p>
<p style="font-size: 20px">Where: <br><br>
. $U$ is the number of questions in the test set.<br><br>
. $P(k)$ is the precision and cutoff $k$. <br><br>
. $n$ is the number of predictions per question. <br><br>
. $rel(k)$ is an indicator function equaling $1$ if the item at rank $k$ is a relevant label (i.e., correct), zero otherwise.<br><br>
Once a correct label has been scored for an individual question in the test set, that label is no longer considered relevant for that question, and additional predictions of that label are skipped in the calculation. <br><br>
In the following table, we explore the attributes available and their description.</p>

<table style="font-family: Arial, sans-serif; font-size: 16px;">
  <tr>
    <th><b>Attribute</b></th>
    <th><b>Description</b></th>
  </tr>
  <tr>
    <td><b>Prompt</b></td>
    <td>The text of questions being asked.</td>
  </tr>
  <tr>
    <td><b>A</b></td>
    <td>Option A; if this option is correct, then answer will be A.</td>
  </tr>
  <tr>
    <td><b>B</b></td>
    <td>Option B; if this option is correct, then answer will be B.</td>
  </tr>
  <tr>
      <td><b>C</b></td>
      <td>Option C; if this option is correct, then answer will be C.</td>
    </tr>
  <tr>
      <td><b>D</b></td>
      <td>Option D; if this option is correct, then answer will be D.</td>
    </tr>
  <tr>
      <td><b>E</b></td>
      <td>Option E; if this option is correct, then answer will be E.</td>
    </tr>
  <tr>
      <td><b>Answer</b></td>
      <td>The most correct answer, as defined by the generating LLM (one of A, B, C, D, or E).</td>
    </tr>
  <tr>
</table>


<span style="font-size: 20px">In the next cells of code, we will import relevant libraries and write some helpful functions.</span>

> <span style="font-size: 20px">📚 Since this is my very first time working on a task such as this one, I can't forget to link the following notebooks, which has served as reference for this one: <br><br>
. <a href = "https://www.kaggle.com/code/wlifferth/starter-notebook-ranked-predictions-with-bert">Starter Notebook: Ranked Predictions with BERT</a> <br><br>
. <a href = "https://www.kaggle.com/code/vad13irt/starter-notebook-deberta-base-lr-3e-5-7-epochs">Starter Notebook: deberta base, lr 3e-5, 7 epochs</a></span>

<h1 id = 'intro' style="border-bottom: 1px solid #ccc;
                        padding-bottom: 10px;
                        margin-bottom: 10px;
                        font-size: 28px;
                        font-weight: bold;
                        color: black;
                        font-family: Poppins">- Version Updates -</h1>

<table style="font-family: Poppins, sans-serif; font-size: 12px;">
  <tr>
    <th><b>Version</b></th>
    <th><b>Description</b></th>
  </tr>
  <tr>
    <td><b>Version 10</b></td>
      <td>First attempt. <br> No further comments or much detail. <br> <b>Public LB:</b> 0.620</td>
  </tr>
  <tr>
    <td><b>Version 22</b></td>
      <td>Finished designing markdown cells. <br> Used the <a href = "https://www.kaggle.com/datasets/radek1/additional-train-data-for-llm-science-exam?select=6000_train_examples.csv">📊 6.5k train examples for LLM Science Exam 📝</a> dataset to increase the training data. <br>Changed some parameters for training and introduced a separate validation set. <br> <b>Public LB:</b> 0.623 <br> </td>
  </tr>
  <tr>
    <td><b>Version 23</b></td>
      <td>Used a <u>larger</u> pre-trained model (<a href ="https://www.kaggle.com/datasets/radek1/deberta-v3-large-hf-weights">Deberta v3 large HF weights</a>), aiming at higher scores <br> <b>Public LB:</b> 0.537 </td>
  </tr>
      <tr>
    <td><b>Version 24</b></td>
      <td> I've used similar parameters as the ones in this notebook (<a href = "https://www.kaggle.com/code/mayank00rastogi/deberta-v3-xlargemcq-s-ipynb?scriptVersionId=137592972">deberta-v3-xlargeMCQ's.ipynb</a>).<br> I decreased the extra data, leaving a total of 700 samples to be trained on, instead of 6,700. <br><b>Public LB:</b> 0.662 </td>
  </tr>
    <tr>
    <td><b>Version 27</b></td>
      <td> Tried different parameter settings.<br><b>Public LB:</b> 0.650 <br><br></td>
  </tr>
       <tr>
    <td><b>Version 29</b></td>
      <td> Tried different parameter settings and increased the training data back to 6,700 samples.<br><b>Public LB:</b> 0.631<br><br></td>
  </tr>
       <tr>
    <td><b>Version 31</b></td>
      <td> Tried applying Optuna for Hyperparameter search.<br><b>Public LB:</b> 0.364<br><br></td>
  </tr>
    <tr>
    <td><b>Version 33/34</b></td>
      <td> Once again, I've reduced the size of the data back to 700 samples.<br><br>I changed some parameters once again, choosing values similar to the ones in <a href = "https://www.kaggle.com/code/radek1/new-dataset-deberta-v3-large-training">this notebook.</a> <br><b>Public LB: 0.702</b> </td>
  </tr>
       <tr>
    <td><b>Version 37</b></td>
      <td> Added the Wikipedia-Stem-1K data for training.<br><br>Used the weights for Deberta uploaded by HYC.<br><b>Public LB: 0.739</b> </td>
  </tr>
    <tr>
    <td><b>Version 39</b></td>
      <td> Added the 60k data with context for training.<br><br>Used the weights for Deberta uploaded by HYC.<br><b>Public LB: 0.748</b><br><br><br><b> ** Best Public LB so far **</b> </td>
  </tr>
</table>

In [ ]:
# Importing Libraries

# Data Handling
import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import Optional, Union
from datasets import Dataset

# Data Visualization
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objs as go
import plotly.subplots as sp
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
from IPython.display import display
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

# Statistics & Mathematics
import scipy.stats as stats
from scipy.stats import shapiro, skew
import math

# RFECV for feature selection
from sklearn.feature_selection import RFECV

# Machine Learning Pipeline & process
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

# Preprocessing data
from sklearn.preprocessing import RobustScaler, StandardScaler, QuantileTransformer, FunctionTransformer
from sklearn.compose import ColumnTransformer

# Model Selection for Cross Validation
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split

# Machine Learning metrics
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, cohen_kappa_score, make_scorer

# ML regressors
from sklearn.linear_model import HuberRegressor,RANSACRegressor, TheilSenRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, StackingRegressor, AdaBoostRegressor, RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# ML classifiers
from sklearn.ensemble import HistGradientBoostingClassifier, AdaBoostClassifier, RandomForestClassifier
from sklearn.ensemble import StackingClassifier, VotingClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Clustering model
from sklearn.cluster import KMeans

# Optuna for tuning models
import optuna

# Randomizer
import random

# Encoder of categorical variables
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

# Importing HuggingFace's Transformers
from transformers import (AutoTokenizer, 
                          AutoModelForMultipleChoice, 
                          Trainer, TrainingArguments, 
                          EarlyStoppingCallback,
                          T5Tokenizer, T5ForConditionalGeneration)
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy

# PyTorch
import torch

# Hiding warnings 
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Configuring Pandas to exhibit larger columns 
'''
This is going to allow us to read the questions and answers
'''
pd.set_option('display.max_colwidth', 1000)

In [ ]:
# Checking if GPU is available
if torch.cuda.is_available():
    print("GPU is available")
    device = torch.device('cuda')
else:
    print("GPU is not available")
    device = torch.device('cpu')

In [ ]:
# Defining seed and the template for plots
seed = 42
plotly_template = 'simple_white'

In [ ]:
def dataframe_description(df):
    """
    This function prints some basic info on the dataset.
    """
    categorical_features = []
    continuous_features = []
    binary_features = []
    
    for col in df.columns:
        if df[col].dtype == object:
            categorical_features.append(col)
        else:
            if df[col].nunique() <= 2:
                binary_features.append(col)
            else:
                continuous_features.append(col)
    
    print("\n{} shape: {}".format(type(df).__name__, df.shape))
    print("\n{:,.0f} samples".format(df.shape[0]))
    print("\n{:,.0f} attributes".format(df.shape[1]))
    print(f'\nMissing Data: \n')
    print(df.isnull().sum())
    print(f'\nDuplicates: {df.duplicated().sum()}')
    print(f'\nData types: \n')
    print(df.dtypes)
    print(f'\nCategorical features: \n')
    if len(categorical_features) == 0:
        print('No Categorical Features')
    else:
        for feature in categorical_features:
            print(feature)
    print(f'\nContinuous features: \n')
    if len(continuous_features) == 0:
        print('No Continuous Features')
    else:
        for feature in continuous_features:
            print(feature)
    print(f'\nBinary features: \n')
    if len(binary_features) == 0:
        print('No Binary Features')
    else:
        for feature in binary_features:
            print(feature)
    print(f'\n{type(df).__name__} Head: \n')
    display(df.head(5))
    print(f'\n{type(df).__name__} Tail: \n')
    display(df.tail(5))

In [ ]:
def plot_correlation(df):
    '''
    This function is resposible to plot a correlation map among features in the dataset
    '''
    corr = np.round(df.corr(), 2)
    mask = np.triu(np.ones_like(corr, dtype = bool))
    c_mask = np.where(~mask, corr, 100)

    c = []
    for i in c_mask.tolist()[1:]:
        c.append([x for x in i if x != 100])
    
    fig = ff.create_annotated_heatmap(z=c[::-1],
                                      x=corr.index.tolist()[:-1],
                                      y=corr.columns.tolist()[1:][::-1],
                                      colorscale = 'bluyl')

    fig.update_layout(title = {'text': '<b>Feature Correlation <br> <sup>Heatmap</sup></b>'},
                      height = 650, width = 650,
                      margin = dict(t=210, l = 80),
                      template = 'simple_white',
                      yaxis = dict(autorange = 'reversed'))

    fig.add_trace(go.Heatmap(z = c[::-1],
                             colorscale = 'bluyl',
                             showscale = True,
                             visible = False))
    fig.data[1].visible = True

    fig.show()

In [ ]:
def describe(df):
    '''
    This function plots a table containing Descriptive Statistics of the Dataframe
    '''
    mean_features = df.mean().round(2).apply(lambda x: "{:,.2f}".format(x)) 
    std_features = df.std().round(2).apply(lambda x: "{:,.2f}".format(x)) 
    q1 = df.quantile(0.25).round(2).apply(lambda x: "{:,.2f}".format(x))
    median = df.quantile(0.5).round(2).apply(lambda x: "{:,.2f}".format(x))
    q3 = df.quantile(0.75).round(2).apply(lambda x: "{:,.2f}".format(x))


    # Generating new Dataframe
    describe_df = pd.DataFrame({'Feature Name': mean_features.index,
                                'Mean': mean_features.values,
                                'Standard Deviation': std_features.values,
                                '25%': q1.values,
                                'Median': median.values,
                                '75%': q3.values})

    # Generating a Table w/ Pyplot
    fig = go.Figure(data = [go.Table(header=dict(values=list(describe_df.columns),
                                                 align = 'center',
                                                 fill_color = 'midnightblue',
                                               font=dict(color = 'white', size = 18)),
                                     cells=dict(values=[describe_df['Feature Name'],
                                                        describe_df['Mean'],
                                                        describe_df['Standard Deviation'],
                                                       describe_df['25%'],
                                                       describe_df['Median'],
                                                       describe_df['75%']],
                                                fill_color = 'gainsboro',
                                                align = 'center'))
                           ])

    fig.update_layout(title = {'text': f'<b>Descriptive Statistics of the Dataframe<br><sup> (Mean, Standard Deviation, 25%, Median, and 75%)</sup></b>'},
                      template = plotly_template,
                      height = 700, width = 950,
                      margin = dict(t = 100))

    fig.show()

In [ ]:
def plot_distplot(df, x):  
    '''
    This function creates a distribution plot for continuous variables
    '''
    
    feature = df[x]

    fig = ff.create_distplot([feature], [x], show_hist=False)

    fig.update_layout(
        title={'text': f'<b>Distplot <br> <sup>{x}</sup></b>',
               'xanchor': 'left',
               'x': 0.05},
        height=600,
        width=1000,
        margin=dict(t=100),
        template= plotly_template,
        showlegend=True
    )

    fig.show()

In [ ]:
def plot_histogram_matrix(df):
    '''
    This function identifies all categorical features within the dataset and plots
    a matrix of histograms for each attribute
    '''
    categorical_features = df.select_dtypes(include=['object']).columns.tolist()
    
    
    df_word_counts = pd.DataFrame()
    
    for feat in categorical_features:
        
        df_word_counts[feat] = df[feat].apply(lambda x: len(str(x).split()))

    num_cols = 2
    num_rows = (len(categorical_features) + 1) // num_cols

    fig = make_subplots(rows=num_rows, cols=num_cols)

    for i, feature in enumerate(categorical_features):
        row = i // num_cols + 1
        col = i % num_cols + 1

        fig.add_trace(
            go.Histogram(
                x=df_word_counts[feature],
                name=feature
            ),
            row=row,
            col=col
        )

        fig.update_xaxes(title_text=feature, row=row, col=col)
        fig.update_yaxes(title_text='Frequency', row=row, col=col)
        fig.update_layout(
            title=f'<b>Histogram Matrix<br> <sup> Word Count per Feature</sup></b>',
            showlegend=False
        )

    fig.update_layout(
        height=350 * num_rows,
        width=1000,
        margin=dict(t=100, l=80),
        template= plotly_template  
    )

    fig.show()

In [ ]:
def plot_boxplot_matrix(df):
    
    '''
    This function identifies all categorical features within the dataset and plots
    a matrix of boxplots for each attribute
    '''
    
    categorical_features = df.select_dtypes(include = ['object']).columns.tolist()
    
    df_word_counts = pd.DataFrame()
    
    for feat in categorical_features:
        df_word_counts[feat] = df[feat].apply(lambda x: len(str(x).split()))
    
    num_cols = 2
    num_rows = (len(categorical_features) + 1) // num_cols


    fig = make_subplots(rows=num_rows, cols=num_cols)


    for i, feature in enumerate(categorical_features):
        row = i // num_cols + 1
        col = i % num_cols + 1

        fig.add_trace(
            go.Box(
                y=df_word_counts[feature],
                name = feature
            ),
            row=row,
            col=col
        )

        fig.update_yaxes(title_text = 'Frequency', row=row, col=col)
        fig.update_xaxes(title_text= feat, row=row, col=col)
        fig.update_layout(
            title=f'<b>Boxplot Matrix<br> <sup> Word Count per Feature</sup></b>',
            showlegend=False,
            yaxis=dict(
            tickangle=-90  
        )
        )

    fig.update_layout(
        height=350 * num_rows,
        width=1000,
        margin=dict(t=100, l=80),
        template= plotly_template
    )


    fig.show()

In [ ]:
def scatterplot(df, x, y):
    '''
    This function takes a dataframe and X and y axes to plot a scatterplot
    '''

    color_dict = {
        0: 'orange',
        1: 'blue',
        2: 'green',
        3: 'red',
        4: 'black',
        5: 'purple',
        6: 'pink',
        7: 'brown',
        8: 'teal',
        9: 'magenta',
        10: 'cyan',
        11: 'olive',
        12: 'navy',
        13: 'indigo',
        14: 'maroon',
        15: 'turquoise',
        16: 'silver',
        17: 'gold'
    }
    
    color_index = random.choice(list(color_dict.keys()))
    color = color_dict[color_index]

    fig = px.scatter(df, y=y, x=x)
    fig.update_traces(marker=dict(size=10, color=color))
    fig.update_layout(
        title={'text': f'<b>Scatterplot <br> <sup>{x} x {y}</sup></b>'},
        height=750,
        width=850,
        margin=dict(t=80, l=80),
        template= plotly_template
    )
    fig.show()

In [ ]:
def clustered_scatterplot(df, y, x, cluster):
    '''
    This function takes a dataframe, x, and y axes to plot a scatterplot colored accordingly to clusters
    It also prints a count of values for each cluster
    '''
    fig = px.scatter(df,
                     y = y,
                     x = x,
                     color = cluster, symbol = cluster)

    fig.update_traces(marker = dict(size = 10))

    fig.update(layout_coloraxis_showscale=False)

    fig.update_layout(title = {'text': f'<b>Clustered Scatterplot <br> <sup> {y} x {x} </sup></b>',
                              'xanchor': 'left',
                              'x': 0.05},
                     height = 600, width = 700,
                     margin = dict(t=100),
                     template = plotly_template,
                     showlegend = True)

    fig.show()

    print('Cluster Count:')
    print(f'{df[cluster].value_counts()}')

In [ ]:
def barplot(df, feat):    
    
    '''
    This function is supposed to organize the n top value counts of any attribute and plot a Barplot
    '''
    
    counts = df[feat].value_counts()
    fig = px.bar(y=counts.values, 
                 x=counts.index, 
                 color = counts.index,
                 text=counts.values)

    fig.update_layout(title=f'<b>Frequency of options in the {feat} variable<br> <sup> Barplot</sup></b>',
                      xaxis=dict(title=f'{feat}'),
                      yaxis=dict(title='Count'),
                      legend=dict(title=f'{feat}'),
                      showlegend=True,
                      height=600,
                      width=1000,
                      margin=dict(t=100, l=80),
                      template= plotly_template)
    fig.show()

In [ ]:
def shapiro_wilk_test(df):
    '''
    This function performs a Shapiro-Wilk test to check if the data is normally distributed or not, as well as skewness
    '''
    print(f'\033[1mShapiro-Wilk Test & Skewness:\033[0m')
    print('\n- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -  \n')

    numeric_columns = df.select_dtypes(include=['float', 'int']).columns

    for feature in numeric_columns:
        stats, p_value = shapiro(df[feature])

        if p_value < 0.05:
            text = f'{feature} Does Not Seem to be Normally Distributed'
        else:
            text = f'{feature} Seems to be Normally Distributed'

        print(f'{feature}')
        print(f'\n  Shapiro-Wilk Statistic: {stats:.2f}')
        print(f'\n  Shapiro-Wilk P-value: {p_value}')
        print(f'\n  Skewness: {np.round(skew(df[feature]), 2)}')
        print(f'\n  Conclusion: {text}')
        print('\n===============================================================================================')

    print('\n- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -  \n')
    print(f'\033[1mEnd of Shapiro-Wilk Test\033[0m')

In [ ]:
def boxplot(df, y, x, color):    
    '''
    This function plots a Y and X boxplot
    '''
    fig = px.box(df, y= y , x = x, color= color)

    fig.update_layout(title=f'<b>Boxplot<br> <sup> {y} by {x}</sup></b>',
                      showlegend=False,
                      yaxis=dict(tickangle= -45),
                      height=600,
                      width=1000,
                      margin=dict(t=100, l=80),
                      template= plotly_template)

    fig.show()

In [ ]:
def pred_vs_true_plot(y_true, y_pred):
    '''
    This function takes values for y_true and y_val, and plots a scatterplot along with a line of best fit
    '''

    slope, intercept = np.polyfit(y_true, y_pred, 1)
    fit_line = slope * y_true + intercept

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=y_true, y=y_pred, mode='markers', name='Data Points'))
    fig.add_trace(go.Scatter(x=y_true, y=fit_line, mode='lines', line=dict(color='red'), name='Fit-line'))
    fig.update_traces(marker=dict(size=10, color='blue'))
    fig.update_layout(
        title={'text': f'<b>True x Predicted <br> <sup>Scatterplot</sup></b>'},
        xaxis=dict(title='True Salaries'), 
        yaxis=dict(title='Predicted Salaries'),
        height=750,
        width=850,
        margin=dict(t=250, l=80),
        template= plotly_template,
    )
    fig.show()


In [ ]:
def three_axes_scatterplot(df, x, y, z):   
    
    '''
    This function takes a dataframe and different attributes to build a 3D scatterplot
    
    '''
    
    scatterplot = go.Scatter3d(
        x= df[x],
        y= df[y],
        z= df[z],  
        mode='markers')

    fig = go.Figure(data=scatterplot)
    fig.update_layout(
        title={'text': f'<b>3D Scatterplot <br> <sup>{x} x {y} x {z}</sup></b>',
               'xanchor': 'left',
               'x': 0.05},
        height=600,
        width=700,
        margin=dict(t=100),
        template= plotly_template,
        showlegend=True
    )

    
    fig.show()

In [ ]:
def violin_boxplot(df, y, x, color):    
    '''
    This function plots a Y and X ridgeline plot
    '''
    
    fig = px.violin(df, y=y, x=x, color=color, box=True, points= 'all')

    fig.update_layout(title=f'<b>Violin Boxplot<br> <sup>{x} by {y}</sup></b>',
                      showlegend=False,
                      yaxis=dict(tickangle=-45),
                      height=600,
                      width=1000,
                      margin=dict(t=100, l=80),
                      template= plotly_template)

    fig.show()

In [ ]:
def individual_boxplot(df, x):    
    fig = px.box(df, x = x)

    fig.update_layout(title=f'<b>Boxplot<br> <sup> {x}</sup></b>',
                      showlegend=False,
                      yaxis=dict(tickangle= -45),
                      height=400,
                      width=1000,
                      margin=dict(t=100, l=80),
                      template= plotly_template)

    fig.show()

In [ ]:
def elbow_curve(wss):    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x = list(range(1,10)),
                            y = wss,
                            mode = 'lines+markers',
                            marker = dict(color = 'midnightblue'),
                            name = 'WSS'))

    
    fig.update_layout(title = {'text': '<b>Elbow Curve Plot <br> <sup>Within-Cluster Sum of Squares</sup></b>'},
                     height = 400, width = 950,
                     xaxis_title = 'Number of Clusters',
                     yaxis_title = 'Within-Cluster Sum of Squares (WSS)',
                     margin = dict(t=80),
                     template = plotly_template)

    fig.show()

In [ ]:
def split_train_test(df, test_size, seed):
    
    '''
    This function splits a dataframe for training and testing according to test_size
    '''
    
    train, test = train_test_split(df, test_size = test_size, shuffle = True, random_state = seed) # Splitting data

    print(f'\n Train shape: {train.shape}\n')
    print(f'\n {len(train)} Samples \n')
    print(f'\n {len(train.columns)} Attributes \n')
    display(train.head(10))
    print('\n' * 2)

    print(f'\n Test shape: {test.shape:}\n')
    print(f'\n {len(test)} Samples \n')
    print(f'\n {len(test.columns)} Attributes \n')
    display(test.head(10))
    
    return train, test

In [ ]:
def X_y_split(df, target_variable):
    
    '''
    This function takes a dataframe and a target variable to create an X (predictors) dataframe and a y Series
    '''
    
    X, y = df.drop([target_variable], axis = 1), df[target_variable] 

    #Printing info on X and y
    print(f'\nX shape: {X.shape}\n')
    print(f'\n{len(X)} Samples \n')
    print(f'\n{len(X.columns)} Attributes \n')
    display(X.head(10))
    print('\n')
    print(f'\ny shape: {y.shape}\n')
    print(f'\n{len(y)} Samples \n')
    display(y.head(10))
    
    return X, y

<h1 id = 'eda' style="border-bottom: 1px solid #ccc;
                        padding-bottom: 10px;
                        margin-bottom: 10px;
                        font-size: 38px;
                        font-weight: bold;
                        color: black;
                        font-family: Poppins">Exploratory Data Analysis</h1>

<p style="font-size: 20px">We start our analysis by loading the dataset and observing some general behaviors across the entire data. <br><br>
Afterwards, we explore the variables according to their type (continuous, binary, etc.), and also explore the target variable separately.</p>

In [ ]:
df = pd.read_csv('/kaggle/input/kaggle-llm-science-exam/train.csv') # Loading data
dataframe_description(df) # Printing info on the data

> <p style="font-size: 20px"><b>📝 We have a total of 200 questions. <br><br>
    📝 All features, except for <code>id</code>, are categorical features.</b></p>

In [ ]:
df = df.drop('id', axis = 1)  # Dropping 'Id' columns

<h1 id = 'eda2' style="border-bottom: 1px solid #ccc;
                        padding-bottom: 10px;
                        margin-bottom: 10px;
                        font-size: 26px;
                        font-weight: bold;
                        color: black;
                        font-family: Poppins">Continuous Features</h1>

<p style="font-size: 20px">Let's start our EDA by observing the <mark style="background-color: yellow;"><b>frequency of words</b></mark> in each variable.</p>

In [ ]:
# Creating a copy dataframe consisting of the independent features
X_copy = df.copy().drop('answer', axis = 1)

In [ ]:
plot_histogram_matrix(X_copy)

In [ ]:
# Creating a new dataframe containing counts of words
word_counts = pd.DataFrame()
for i in X_copy.columns.tolist():
    word_counts[i] = X_copy[i].apply(lambda x: 
                                     len(str(x).split())) # Obtaining the number of words in each string in the column of the dataframe

fig = go.Figure()
for i, col in enumerate(word_counts.columns):
    fig.add_trace(go.Box(y=word_counts[col], name=col, 
                         marker_color=px.colors.qualitative.Plotly[i]))

fig.update_yaxes(title_text = 'Frequency')
fig.update_xaxes(title_text= 'Features')

fig.update_layout(
    title='<b>Boxplots<br> <sup> Word Count per Feature</sup></b>',
    showlegend=False,
    yaxis=dict(tickangle=-45),
    height=600,
    width=1000,
    margin=dict(t=100, l=80),
    template=plotly_template)

fig.show()

> <p style="font-size: 20px"><b>📝 <code>prompt</code>, which is the variable containing <mark style="background-color: yellow;"><b>questions</b></mark>, has the lower frequency of words among the features. <br><br>
    📝 All answer options have a very similar number of word counts on average. The larger outlier is in <code>B</code>.</b></p>

<p style="font-size: 20px">We can also use word clouds for each attribute. This helps us to enhance our understandment of the content of the dataframe, as well as observing the most frequent words in each attribute. </p>

In [ ]:
# For each column in the DataFrame, we concatenate all its values into a single string, separated by spaces.
for i in X_copy.columns.tolist():
    text = ' '.join(str(v) for v in X_copy[i])
    
    # Generating clouds of words
    wc = WordCloud(width = 1300,
                  height = 800,
                  background_color = 'white').generate(text)

    plt.figure(figsize = (10, 15))
    plt.imshow(wc, interpolation='bilinear')
    plt.title(f'\nWord Cloud for {i}')
    plt.axis("off")
    plt.show()

> <p style="font-size: 20px"><b>📝 It's possible to identify words related to science and physics, such as <code>particule</code>,<code>energy</code>, <code>light</code>, <code>mass</code>, <code>density</code>, <code>energy</code>, etc.</b></p>

<h1 id = 'eda2' style="border-bottom: 1px solid #ccc;
                        padding-bottom: 10px;
                        margin-bottom: 10px;
                        font-size: 26px;
                        font-weight: bold;
                        color: black;
                        font-family: Poppins">Answer - Target Variable</h1>

<p style="font-size: 20px">We may also use a <mark style="background-color: yellow;"><b>barplot</b></mark> to identify the frequence of options in the <code>answer</code> varibale. This is going to help us identify the options that are most frequently selected as the correct answer for the questions in the dataset.</p>

In [ ]:
barplot(df, 'answer')

> <p style="font-size: 20px"><b>📝 Most right answers are belonging to option <code>B</code>, followed by option <code>C</code>.</b></p>

<h1 id = 'model' style="border-bottom: 1px solid #ccc;
                        padding-bottom: 10px;
                        margin-bottom: 10px;
                        font-size: 38px;
                        font-weight: bold;
                        color: black;
                        font-family: Poppins">Modeling</h1>

<p style="font-size: 20px">In attempt to improve accuracy, we are going to add some extra data to our original dataframe.<br><br>
I am going to use both <i>csv</i> files avilable on the <a href = "https://www.kaggle.com/datasets/radek1/additional-train-data-for-llm-science-exam?select=6000_train_examples.csv">📊 6.5k train examples for LLM Science Exam 📝</a> dataset.</p>

In [ ]:
# Concatenating original dataframe to extra dataframes
augmented_df = pd.concat([
    df,
    pd.read_csv('/kaggle/input/60k-data-with-context-v2/all_12_with_context2.csv') # 60k data with context
    #pd.read_csv('/kaggle/input/additional-train-data-for-llm-science-exam/6000_train_examples.csv'), # Loading 6000 extra samples
    #pd.read_csv('/kaggle/input/additional-train-data-for-llm-science-exam/extra_train_set.csv'), # Loading 500 extra samples
    #pd.read_csv('/kaggle/input/llm-science-3k-data/test.csv'), # Adding 3,233 more samples
    #pd.read_csv('/kaggle/input/wikipedia-stem-1k/stem_1k_v1.csv'), # Adding wikipedia stem 1k dataset

])

augmented_df = augmented_df.drop(columns = 'source') # Removing 'source' column
augmented_df = augmented_df.fillna('').sample(1_024) # Filling NANs
augmented_df.reset_index(inplace = True, drop = True) # Reseting index
print('\nAugmented Data: \n')
display(augmented_df.head(5))
print('\n* * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * \n')
display(augmented_df.tail(5))
print(f'\nSamples = {augmented_df.shape[0]}')

> <p style="font-size: 20px"><b>📝 We went from 200 samples to 7,700 samples.</b></p>

<p style="font-size: 20px">Now I'm going to split the data, creating a dataframe for training and another for validation. I am going to use 70% of the data for training and 30% for validation.</p>

In [ ]:
# Creating training and validation sets
train_df, val_df = train_test_split(augmented_df, 
                                    test_size=0.3, shuffle = True, random_state=seed)

<p style="font-size: 20px">Now we convert the dataframes to Datasets.</p>

In [ ]:
# Converting dataframes into datasets
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

print('\nTrain Dataset:\n')
print(train_ds)
print('\nValidation Dataset:\n')
print(val_ds)

<p style="font-size: 20px">We are now going to use the <code>AutoTokenizer</code> class from Hugging Face, as well as the <code>from_pretrained()</code> method, to load the token vocabulary pretrained for the de DeBERTa model.</p>

In [ ]:
# Instantiating DeBERTa v3 Tokenizer, which will transform text into tokens
tokenizer = AutoTokenizer.from_pretrained('/kaggle/input/2023kagglellm-deberta-v3-large-model1')

<p style="font-size: 20px">Now we creat two lists, <code>options</code> and <code>indices</code>, where the first will consist of the characters  <i>A</i>, <i>B</i>, <i>C</i>, <i>D</i>, <i>E</i>, while the latter will consist of the integers <i>0</i>, <i>1</i>, <i>2</i>, <i>3</i>, <i>4</i>. <br><br>
Following on, we create the <code>option_to_index</code> and <code>index_to_option</code> dictionaires to map each option to an index, such as they look like this: { ${0:A}$,   ${1:B}$,   ${2:C}$,   ${3:D}$,   ${4:E}$ }. <br><br>
The <code>preprocess</code> function will be responsible for returning the tokens in a <i>"language"</i> that our model understands.</p>

In [ ]:
# Setting up answer choices and their respective indices
options = 'ABCDE'
indices = list(range(5)) # Indexing 0, 1, 2, 3, 4 for each option


option_to_index = {option: index for option, index in zip(options, indices)} # Converting options to indices '''A to 0'''
index_to_option = {index: option for option, index in zip(options, indices)} # Converting indices to options '''0 to A'''


def preprocess(example):
    first_sentence = [example['prompt']] * 5 # Repeating the same question 5 times
    second_sentence = [] # Creating list of possible answers
    for option in options:
        second_sentence.append(example[option])
        
    # The tokenizer converts the question and answers into 'tokens'.
    # 'tokens' are simply a sequence of integers in which each specific integer corresponds to 
    # a word or subword that BERT is capable of comprehending 
    tokenized_example = tokenizer(first_sentence, second_sentence, truncation=True)
    
    # Indexing label - A,B,C,D or E - to either 0, 1, 2, 3, 4, or 5 
    tokenized_example['label'] = option_to_index[example['answer']]
    
    # tokenized_example returns:
        # input_ids --> List of lists represents the tokens 
        # token_type_ids --> A list of lists indicating whether each token belongs to the 1st or 2nd sentence
        # attention_mask --> List of list where each inner list is either 1 or 0. 1 are not padding tokens, 0 are padding tokens
        # label --> The index for the correct option
    return tokenized_example

In [ ]:
# Tokenizing train Dataset
tokenized_train_ds = train_ds.map(preprocess, batched=False, 
                                  remove_columns=['prompt', 'A', 'B', 'C', 'D', 'E', 'answer', '__index_level_0__'])
print(tokenized_train_ds)

In [ ]:
# Tokenizing validation Dataset
tokenized_val_ds = val_ds.map(preprocess, batched=False, 
                                  remove_columns=['prompt', 'A', 'B', 'C', 'D', 'E', 'answer', '__index_level_0__'])
print(tokenized_val_ds)

> <p style="font-size: 20px"><b>📝 We now have two tokenized datasets consisting of the following variables: $input$_$ids$, $token$_$type$_$ids$, $attention$_$mask$, and $label$.</b></p>

<p style="font-size: 20px">In the following class, we are going to format and batch the data.</p>

In [ ]:
@dataclass
class DataCollatorForMultipleChoice:
    '''
    This class is designed to handle the formatting and batching the data for mutiple-choice tasks
    '''
    
    # The tokenizer to be used for tokenizing the data
    tokenizer: PreTrainedTokenizerBase
    
    # The strategy to be used for padding the data
    padding: Union[bool, str, PaddingStrategy] = True
    
    # The maximum length for any input sequence
    max_length: Optional[int] = None
    
    # If provided, pad the sequences to a multiple of this value
    pad_to_multiple_of: Optional[int] = None
    
    def __call__(self, features):
        # Finding the correct label key in the features
        label_name = "label" if 'label' in features[0].keys() else 'labels'
        
        # Extracting the labels and removing them from features
        labels = [feature.pop(label_name) for feature in features]
        
        # Obtaining batch size
        batch_size = len(features)
        
        # Obtaining number of choices
        num_choices = len(features[0]['input_ids'])
        
        # Reestructuring features so each question-choice pair becomes a separate example
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features
        ]
        flattened_features = sum(flattened_features, [])
        
        # Padding all sequences to the same length
        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors='pt',
        )
        
        # Reshaping the batch back into the original format
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        
        # Adding the labels back into the batch as a tensor
        batch['labels'] = torch.tensor(labels, 
                                       dtype=torch.int64)
        
        # Returning the batch
        return batch

<p style="font-size: 20px">We can now load the DeBERTa model for Multiple-Choice task, as well as visualize its architecture.</p>

In [ ]:
# Instantiating model
model = AutoModelForMultipleChoice.from_pretrained('/kaggle/input/2023kagglellm-deberta-v3-large-model1')

In [ ]:
model = model.to(device) # GPU

In [ ]:
print(model) # Printing model's architecture

<p style="font-size: 20px">Now we define some functions to compute the $MAP@3$ score, map the outputs, and compute metrics.</p>

In [ ]:
def competition_score(y_true, y_pred):
    
    """
    Obtaining score for the model
    """
    
    ap_at_3 = 0.0
    for i in range(len(y_true)):
        if y_true[i] in y_pred[i][:3]:
            ap_at_3 += 1
    map3 = ap_at_3 / len(y_true)
    return map3

In [ ]:
def predictions_to_map_output(predictions):
    sorted_answer_indices = np.argsort(-predictions) # Sortting indices in descending order
    top_answer_indices = sorted_answer_indices[:,:3] # Taking the first three indices for each row
    top_answers = np.vectorize(index_to_option.get)(top_answer_indices) # Transforming indices to options - i.e., 0 --> A
    return np.apply_along_axis(lambda row: ' '.join(row), 1, top_answers)

In [ ]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    y_pred = predictions_to_map_output(logits)
    y_true = [index_to_option[label] for label in labels]
    return {'Map@3': np.round(competition_score(y_true, y_pred), 3)}

<p style="font-size: 20px">We use the <code>TrainingArguments</code> class to define some parameters for our model, and then <code>Trainer</code> to put it all together (the model, the parameters, the datasets, etc.)</p>

In [ ]:
#model_dir = 'finetuned_bert'
#def objective(trial):
    # Suggest values for the hyperparameters:
   # learning_rate = trial.suggest_float('learning_rate', 1e-6,1e-3, log = True)
   # per_device_train_batch_size = trial.suggest_categorical('per_device_train_batch_size', [2, 4, 8, 16])
  #  per_device_eval_batch_size = trial.suggest_categorical('per_device_eval_batch_size', [2, 4, 8, 16])

    # Use the suggested hyperparameters in the training arguments:
   # training_args = TrainingArguments(
   #     output_dir=model_dir,
  #      evaluation_strategy="epoch",
  #      save_strategy="epoch",
  #      load_best_model_at_end=True,
  #      save_total_limit=2,
  #      learning_rate=learning_rate,
  #      per_device_train_batch_size=per_device_train_batch_size,
  #      per_device_eval_batch_size=per_device_eval_batch_size,
  #      gradient_accumulation_steps = 2,
  #      gradient_checkpointing = True,
   #     num_train_epochs=7,
   #     weight_decay=0.01,
   #     metric_for_best_model = 'eval_loss',
   #     report_to='none',
   # )

    # Define the trainer with the training arguments and the suggested hyperparameters:
   # trainer = Trainer(
   #     model=model,
   #     args=training_args,
   #     train_dataset=tokenized_train_ds,
   #     eval_dataset=tokenized_val_ds,
   #     tokenizer=tokenizer,
   #     data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
   #     compute_metrics = compute_metrics,
   #     callbacks = [EarlyStoppingCallback(early_stopping_patience = 3,
   #                                       early_stopping_threshold = 0.01)]
   # )

    # Train the model:
   # trainer.train()

    # Evaluate the model:
   # eval_result = trainer.evaluate()

    # Return the evaluation metric you want to optimize:
   # return eval_result["eval_loss"]

In [ ]:
#study = optuna.create_study(direction="minimize")
#study.optimize(objective, n_trials=5)

In [ ]:
#best_params = study.best_params
#print(f'\n Best Params = {best_params} \n')

In [ ]:
model_dir = 'finetuned_model' # Directory to save model and files

# Defining parameters
training_args = TrainingArguments(
    output_dir=model_dir,
    warmup_ratio = 0.8,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=2,
    learning_rate= 5e-6,
    per_device_train_batch_size= 1,
    per_device_eval_batch_size= 2,
    #gradient_accumulation_steps = 2,
    gradient_checkpointing = True,
    num_train_epochs=20,
    weight_decay=0.01,
    metric_for_best_model = 'eval_loss',
    report_to='none',
)

In [ ]:
# Defining Trainer to train the model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_ds,
    eval_dataset=tokenized_val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
    compute_metrics = compute_metrics,
    callbacks = [EarlyStoppingCallback(early_stopping_patience = 2,
                                      early_stopping_threshold = 0.01)]
)

In [ ]:
# Training model
trainer.train()

<p style="font-size: 20px">We may go ahead and predict answers on the tokenized validation data.</p>

In [ ]:
predictions = trainer.predict(tokenized_val_ds) # Predicting answers
predictions_to_map_output(predictions.predictions) # Obtaining the top 3 answers for each question 

<p style="font-size: 20px">Now we load and prepare the <code>test</code> dataframe to predict the answers for its prompts.</p>

In [ ]:
# Loading and visualizing test data
test = pd.read_csv('/kaggle/input/kaggle-llm-science-exam/test.csv')
test.head()

In [ ]:
# Creating an 'answer' column just to make predictions directly with the trainer
test['answer'] = 'A'
test_ds = Dataset.from_pandas(test)
tokenized_test_ds = test_ds.map(preprocess, batched=False, 
                                remove_columns=['prompt', 'A', 'B', 'C', 'D', 'E', 'answer'])

In [ ]:
test_predictions = trainer.predict(tokenized_test_ds) # Predicting the test data

In [ ]:
# Loading df for sample submission
sample_submission = pd.read_csv('/kaggle/input/kaggle-llm-science-exam/sample_submission.csv')
sample_submission

In [ ]:
# Creating new Dataframe containing 'Ids' from 'test' and the respective predictions for each ID
submission = pd.DataFrame({
    'id': test['id'],
    'prediction': predictions_to_map_output(test_predictions.predictions)
})
submission

In [ ]:
# Saving df as csv
submission.to_csv('submission.csv', index=False)

<h1 id = 'conclusion' style="border-bottom: 1px solid #ccc;
                        padding-bottom: 10px;
                        margin-bottom: 10px;
                        font-size: 38px;
                        font-weight: bold;
                        color: black;
                        font-family: Poppins">Conclusion</h1>

<p style='font-size: 20px'>In our initial attempts into this competition, we conducted an EDA using Plotly. This allowed us to identify patterns within the original dataset, providing us insights into the overall structure of the data. <br><br>
Our preliminary submissions were generated by a model that was trained on the pre-trained DeBERTa model. The idea behind this approach is to harness the robust capabilities of a pretrained model, aiming at enhancing the accuracy of predictions.<br><br>
To improve performance, we incorporated additional data into the original training set provided by the competition. This was done with the intention of increasing diversity and volume of data to expose the model to.<br><br>
There are still some extra steps we could take further improve performance. These include:<br><br><br>
🖊️ Fine-tuning parameters. <br><br>
🖊️ Incorporating more diverse data. <br><br>
🖊️ Trying different pre-trained models.<br><br>
🖊️ Implementing ensemble methods.<br><br>
<br><br><br>
All these listed steps can be taken in upcoming versions of this notebook.
<br><br><br><br>
Thank you for reading.</p>

#### <hr style="border: 0; height: 1px; border-top: 0.85px solid #b2b2b2">
<div style="text-align: left; color: #8d8d8d; padding-left: 15px; font-size: 14.25px;">
    Luis Fernando Torres, 2023 <br><br>
    Let's connect!🔗<br>
    <a href="https://www.linkedin.com/in/luuisotorres/">LinkedIn</a> • <a href="https://medium.com/@luuisotorres">Medium</a> • <a href = "https://huggingface.co/luisotorres">Hugging Face</a><br><br>
</div>